In [ ]:
import numpy as np
import torch
from torchvision import datasets, transforms
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import gc
from optimizer import OrthogonalOptimizer
import os

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset  = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(dataset=test_dataset, batch_size=1000, shuffle=False)

In [ ]:
class mlp(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 10)
        )

    def forward(self, x):
        return self.layers(x)

model = mlp().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = OrthogonalOptimizer(model.parameters(), num_bits=3)
gc.collect()

In [ ]:
# dim = 784
# random_matrix = torch.randn(dim, dim)
# rot, _ = torch.linalg.qr(random_matrix)
# rot = rot.to(device)

In [ ]:
epochs = 500
train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []
checkpoint_path = "mnist_checkpoint.pt"
start_epoch = 0

In [ ]:
if os.path.exists(checkpoint_path):
    print(f"Found checkpoint '{checkpoint_path}', resuming training")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    train_losses = checkpoint['train_losses']
    train_accuracies = checkpoint['train_accuracies']
    test_losses = checkpoint['test_losses']
    test_accuracies = checkpoint['test_accuracies']
    print(f"Resuming from loop index {start_epoch}, epoch {start_epoch + 1}")
else:
    print("No checkpoint found, starting training from scratch.")

In [ ]:
for epoch in range(start_epoch, epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        # # applying random orthogonal rotation to inputs to makes pixels incoherent
        # inputs = inputs.view(inputs.size(0), -1)
        # inputs = inputs @ rot

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    print(f'Epoch [{epoch + 1}/{epochs}]')
    print("Training loss", (running_loss / total))  
    train_losses.append(running_loss / total)
    print("Training accuracy", (correct *100/ total),'%')
    train_accuracies.append(correct * 100 / total)

    model.eval()
    running_loss_eval = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            # inputs = inputs.view(inputs.size(0), -1)
            # inputs = inputs @ rot

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss_eval += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print("Testing loss", (running_loss_eval / total))
    test_losses.append(running_loss_eval / total)
    print("Testing accuracy", (correct*100 / total),'%')
    test_accuracies.append(correct * 100 / total)

    checkpoint_data = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_losses': train_losses,
        'train_accuracies': train_accuracies,
        'test_losses': test_losses,
        'test_accuracies': test_accuracies
    }
    tmp_checkpoint_path = checkpoint_path + ".tmp"
    torch.save(checkpoint_data, tmp_checkpoint_path)
    os.replace(tmp_checkpoint_path, checkpoint_path)
    print(f"Checkpoint saved securely at end of epoch {epoch + 1}")
    
    gc.collect()